# Day 3: Building & Testing the Memory Module
**Goal:** Implement MemoryBuffer + MemoryAttention, verify tensor shapes, run first memory-augmented prediction.

Today you'll:
1. Understand exactly how your memory module works
2. Test each component individually
3. Run the full memory-augmented forward pass
4. Compare outputs with vs without memory

---
## 0. Setup

In [ ]:
!pip install numpy==2.2.6 -q
# After this cell, go to Runtime > Restart runtime, then continue from the next cell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Research/memory-brain-encoding'
CACHE_DIR = os.path.join(PROJECT_DIR, 'cache')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')

for d in [CACHE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted!')

In [ ]:
%%capture
!pip install git+https://github.com/facebookresearch/tribev2.git

In [ ]:
# Clone YOUR repo to get the memory module
!git clone https://github.com/Mrabbi3/memory-augmented-brain-encoding.git /content/my_repo 2>/dev/null || echo 'Already cloned'

import sys
sys.path.insert(0, '/content/my_repo/src')

# HuggingFace auth
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=hf_token)
except Exception:
    from huggingface_hub import login
    login()

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1. Load TRIBE v2

In [ ]:
from tribev2 import TribeModel
import numpy as np

print('Loading TRIBE v2...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE_DIR)
brain_model = model._model
print(f'Model loaded on {brain_model.device}')
print(f'Hidden dim: {brain_model.config.hidden}')

---
## 2. Import and Understand the Memory Module

Let's import your memory module and test each component step by step.

In [ ]:
# Import your memory module
from memory import MemoryBuffer, MemoryAttention, MemoryAugmentedEncoder
from memory import SlidingWindowBuffer, HierarchicalSummaryBuffer, RandomRetrievalBuffer

print('All memory components imported successfully!')

---
## 3. Test the MemoryBuffer

The MemoryBuffer stores mean-pooled latent vectors from past windows
and retrieves the most similar ones via cosine similarity.

In [ ]:
# Create a buffer
buffer = MemoryBuffer(buffer_size=50, hidden_dim=1152, top_k=5)
print(f'Created: {buffer}')

# Simulate storing 10 past windows
# Each window's transformer output would be [B=1, T=200, H=1152]
print('\nStoring 10 simulated windows...')
for i in range(10):
    fake_window = torch.randn(1, 200, 1152)  # [B, T, H]
    buffer.store(fake_window)
    print(f'  Window {i+1} stored. Buffer size: {buffer.size}')

print(f'\nFinal: {buffer}')

In [ ]:
# Test retrieval
query = torch.randn(1, 1152)  # [B, H] - mean-pooled current window

retrieved = buffer.retrieve(query)
print(f'Query shape: {query.shape}')
print(f'Retrieved shape: {retrieved.shape}')  # Should be [1, 5, 1152]
print(f'  → {retrieved.shape[1]} memories retrieved, each {retrieved.shape[2]} dimensions')

# Verify shapes are correct for cross-attention
assert retrieved.shape == (1, 5, 1152), f'Wrong shape: {retrieved.shape}'
print('\nShape check PASSED!')

In [ ]:
# Test edge cases
print('Testing edge cases...')

# Empty buffer should return None
empty_buffer = MemoryBuffer(buffer_size=50, hidden_dim=1152, top_k=5)
result = empty_buffer.retrieve(query)
assert result is None, 'Empty buffer should return None'
print('  Empty buffer → None ✓')

# Buffer with fewer items than top_k
small_buffer = MemoryBuffer(buffer_size=50, hidden_dim=1152, top_k=5)
small_buffer.store(torch.randn(1, 200, 1152))  # Store just 1
small_buffer.store(torch.randn(1, 200, 1152))  # Store just 2
result = small_buffer.retrieve(query)
print(f'  Buffer with 2 items, top_k=5 → retrieved {result.shape[1]} items ✓')

# FIFO eviction
tiny_buffer = MemoryBuffer(buffer_size=3, hidden_dim=1152, top_k=2)
for i in range(5):
    tiny_buffer.store(torch.randn(1, 200, 1152))
assert tiny_buffer.size == 3, f'Should be 3 after FIFO, got {tiny_buffer.size}'
print(f'  FIFO eviction: stored 5 in buffer of 3 → size={tiny_buffer.size} ✓')

print('\nAll edge cases PASSED!')

---
## 4. Test the MemoryAttention

The MemoryAttention layer performs cross-attention where:
- Q = current window latents [B, T, 1152]
- K, V = retrieved memories [B, K, 1152]
- Output = x + gate * attention(Q, K, V)

In [ ]:
# Create memory attention on the same device as the model
device = brain_model.device
mem_attn = MemoryAttention(hidden_dim=1152, num_heads=8).to(device)

# Count parameters
attn_params = sum(p.numel() for p in mem_attn.parameters())
print(f'MemoryAttention parameters: {attn_params:,}')
print(f'  (This is {attn_params/177205397*100:.2f}% of TRIBE v2\'s 177M params)')

# Check initial gate value
gate_val = torch.tanh(mem_attn.gate).item()
print(f'\nInitial gate value: {gate_val:.4f}')
print('  (Starts near 0 = memory has no influence initially)')
print('  (Will learn to increase during training if memory helps)')

In [ ]:
# Test forward pass with memories
x = torch.randn(1, 200, 1152).to(device)           # Current window [B, T, H]
memories = torch.randn(1, 5, 1152).to(device)       # Retrieved memories [B, K, H]

output = mem_attn(x, memories)

print(f'Input shape:    {x.shape}')
print(f'Memories shape: {memories.shape}')
print(f'Output shape:   {output.shape}')
assert output.shape == x.shape, 'Output must have same shape as input!'
print('\nShape check PASSED! Output shape matches input shape.')

# Verify that with gate=0, output ≈ input
diff = (output - x).abs().mean().item()
print(f'\nMean difference from input: {diff:.6f}')
print('  (Should be very small since gate starts at 0)')

In [ ]:
# Test forward pass WITHOUT memories (should return input unchanged)
output_no_mem = mem_attn(x, None)

diff_no_mem = (output_no_mem - x).abs().max().item()
print(f'Difference when memories=None: {diff_no_mem:.6f}')
assert diff_no_mem == 0.0, 'With no memories, output should exactly equal input'
print('PASSED! No memories → output identical to input.')

---
## 5. Test the Full MemoryAugmentedEncoder

Now let's wrap the TRIBE v2 model with our memory module and run
a prediction on an actual video to verify the complete pipeline works.

In [ ]:
# Create the memory-augmented encoder
mem_encoder = MemoryAugmentedEncoder(
    brain_model=brain_model,
    buffer_size=100,
    top_k=5,
    num_heads=8,
)

print('MemoryAugmentedEncoder created!')
print(f'  Buffer capacity: {mem_encoder.memory_buffer.buffer_size} windows')
print(f'  Retrieval top_k: {mem_encoder.memory_buffer.top_k}')
print(f'  Hidden dim: {mem_encoder.hidden_dim}')
print(f'  Memory attention params: {sum(p.numel() for p in mem_encoder.memory_attention.parameters()):,}')
print(f'\nMemory stats: {mem_encoder.get_memory_stats()}')

In [ ]:
# Generate a test video and run predictions with memory
import os
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

video_path = os.path.join(DATA_DIR, 'test_30s.mp4')
if not os.path.exists(video_path):
    # Create a 30-second test video (longer to test multiple windows)
    os.system(f'ffmpeg -f lavfi -i testsrc=duration=30:size=320x240:rate=24 '
              f'-f lavfi -i sine=frequency=440:duration=30 -shortest -y {video_path} 2>/dev/null')
    print(f'Created 30s test video: {video_path}')
else:
    print(f'Test video exists: {video_path}')

In [ ]:
# Run VANILLA TRIBE v2 (no memory)
print('Running VANILLA TRIBE v2 (baseline)...')
df = model.get_events_dataframe(video_path=video_path)
preds_vanilla, segments = model.predict(events=df)
print(f'Vanilla predictions: {preds_vanilla.shape}')
print(f'  → {preds_vanilla.shape[0]} timepoints × {preds_vanilla.shape[1]} vertices')

In [ ]:
# Now run WITH memory
# We need to manually process windows sequentially
print('\nRunning MEMORY-AUGMENTED TRIBE v2...')

# Reset memory
mem_encoder.reset_memory()
print(f'Memory reset. Buffer: {mem_encoder.memory_buffer}')

# Get the dataloader
loader = model.data.get_loaders(events=df, split_to_build='all')['all']

preds_memory = []
with torch.inference_mode():
    for batch_idx, batch in enumerate(loader):
        batch = batch.to(brain_model.device)
        
        # Forward pass WITH memory
        y_pred = mem_encoder.forward_with_memory(batch)
        preds_memory.append(y_pred.detach().cpu().numpy())
        
        stats = mem_encoder.get_memory_stats()
        print(f'  Batch {batch_idx}: buffer={stats["buffer_size"]}/{stats["buffer_capacity"]}, '
              f'gate={stats["gate_value"]:.4f}')

import numpy as np
from einops import rearrange
preds_memory = np.concatenate(preds_memory, axis=0)
# Reshape to match vanilla format: (B*T, vertices)
if preds_memory.ndim == 3:
    preds_memory = rearrange(preds_memory, 'b d t -> (b t) d')

print(f'\nMemory predictions: {preds_memory.shape}')
print(f'Vanilla predictions: {preds_vanilla.shape}')

---
## 6. Compare Vanilla vs Memory-Augmented Predictions

In [ ]:
import matplotlib.pyplot as plt
from nilearn import plotting, datasets

# Compare predictions
min_t = min(preds_vanilla.shape[0], preds_memory.shape[0])
vanilla_trimmed = preds_vanilla[:min_t]
memory_trimmed = preds_memory[:min_t]

# Compute per-vertex difference
diff = np.abs(memory_trimmed - vanilla_trimmed).mean(axis=0)  # [vertices]

print(f'Mean absolute difference: {diff.mean():.6f}')
print(f'Max absolute difference:  {diff.max():.6f}')
print(f'\nNote: Difference should be SMALL since the gate starts at 0.')
print('The gate will learn to increase during training if memory helps.')

# Correlation between vanilla and memory predictions
from scipy.stats import pearsonr
r_vals = []
for v in range(0, vanilla_trimmed.shape[1], 100):  # Sample every 100th vertex
    if vanilla_trimmed[:, v].std() > 0 and memory_trimmed[:, v].std() > 0:
        r, _ = pearsonr(vanilla_trimmed[:, v], memory_trimmed[:, v])
        r_vals.append(r)

print(f'\nCorrelation between vanilla and memory predictions:')
print(f'  Mean R: {np.mean(r_vals):.4f}')
print(f'  Min R:  {np.min(r_vals):.4f}')
print(f'  (Should be near 1.0 since gate ≈ 0 before training)')

In [ ]:
# Visualize where the differences are on the brain surface
fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
n_verts = diff.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5), subplot_kw={'projection': '3d'})

plotting.plot_surf_stat_map(
    fsaverage['pial_left'], diff[:n_verts],
    hemi='left', view='lateral', colorbar=True, cmap='Reds',
    title='Memory impact (left)', axes=axes[0], figure=fig)

plotting.plot_surf_stat_map(
    fsaverage['pial_right'], diff[n_verts:],
    hemi='right', view='lateral', colorbar=True, cmap='Reds',
    title='Memory impact (right)', axes=axes[1], figure=fig)

plt.suptitle('Difference between vanilla and memory-augmented predictions\n'
             '(Before training — differences are minimal due to gate=0)', y=1.02)
plt.tight_layout()

fig_path = os.path.join(RESULTS_DIR, 'memory_vs_vanilla_diff.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved to {fig_path}')
plt.show()

---
## 7. Test Alternative Memory Strategies

In [ ]:
# Test all three alternative strategies
print('Testing alternative memory strategies...')
print()

strategies = {
    'Retrieval (cosine)': MemoryBuffer(buffer_size=50, hidden_dim=1152, top_k=5),
    'Sliding Window': SlidingWindowBuffer(n_windows=5, hidden_dim=1152),
    'Hierarchical Summary': HierarchicalSummaryBuffer(hidden_dim=1152, decay=0.9),
    'Random Retrieval': RandomRetrievalBuffer(buffer_size=50, hidden_dim=1152, top_k=5),
}

# Store 10 windows in each
for name, strategy in strategies.items():
    for i in range(10):
        strategy.store(torch.randn(1, 200, 1152))
    
    # Retrieve
    query = torch.randn(1, 1152)
    result = strategy.retrieve(query)
    
    print(f'{name}:')
    print(f'  Stored: {strategy.size} windows')
    print(f'  Retrieved shape: {result.shape}')
    print()

---
## 8. Save Day 3 Summary

In [ ]:
import json
from datetime import datetime

session = {
    'date': datetime.now().isoformat(),
    'notebook': '03_memory_module',
    'memory_attention_params': int(attn_params),
    'initial_gate_value': float(gate_val),
    'vanilla_shape': list(preds_vanilla.shape),
    'memory_shape': list(preds_memory.shape),
    'mean_diff': float(diff.mean()),
    'strategies_tested': list(strategies.keys()),
    'status': 'completed'
}

log_path = os.path.join(RESULTS_DIR, 'session_log.json')
if os.path.exists(log_path):
    with open(log_path) as f:
        logs = json.load(f)
else:
    logs = []
logs.append(session)
with open(log_path, 'w') as f:
    json.dump(logs, f, indent=2)

print('Day 3 Summary')
print('='*50)
print(f'Memory attention adds {attn_params:,} params ({attn_params/177205397*100:.2f}% of model)')
print(f'Initial gate: {gate_val:.4f} (no memory influence before training)')
print(f'Mean prediction diff: {diff.mean():.6f}')
print(f'Strategies tested: {len(strategies)}')
print(f'\nAll results saved to Google Drive!')

---
## What You Built Today

1. **MemoryBuffer** — stores compressed representations, retrieves via cosine similarity
2. **MemoryAttention** — cross-attention with gated residual connection
3. **MemoryAugmentedEncoder** — wraps TRIBE v2 with memory capabilities
4. **3 alternative strategies** — sliding window, hierarchical summary, random control

Key design decisions:
- **Gate initialized to 0** → model starts as vanilla TRIBE v2
- **Detached memory storage** → no gradient flow through history
- **Residual connection** → memory can only help, not hurt
- **Shape-preserving** → everything downstream works unchanged

**Next (Day 4):** Set up the training loop to actually train the memory module on fMRI data